In [ ]:
# Cell 1 – Setup: mount Drive, define paths, install VibeVoice from Drive, validate folders

import os
from pathlib import Path

# Detect Colab and mount Google Drive if needed
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:  # Not running in Colab
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

# Base directories – assumes your Drive mirrors your local layout
BASE_DIR = Path("/content/drive/MyDrive/Libri_Vibevoice") if IN_COLAB else Path.cwd()
VIBEVOICE_DIR = BASE_DIR / "VibeVoice"
SPEAKERS_DIR = BASE_DIR / "speakers"
PROMPTS_DIR = BASE_DIR / "prompts"
CLONING_DIR = BASE_DIR / "cloning"  # Directory for reference audio files used in cloning

# Where generated audio will be saved (you can customize this)
OUTPUT_BASE_DIR = BASE_DIR / "outputs" / "raw"  # Base outputs directory - batch run IDs will be subdirectories
OUTPUTS_JSON_PATH = BASE_DIR / "outputs" / "raw" / "outputs.json"

# Directory for caching processed speaker embeddings
SPEAKER_CACHE_DIR = BASE_DIR / "speaker_cache"  # Cache directory for processed speaker embeddings

print("BASE_DIR:", BASE_DIR)
print("VIBEVOICE_DIR:", VIBEVOICE_DIR)
print("SPEAKERS_DIR:", SPEAKERS_DIR)
print("PROMPTS_DIR:", PROMPTS_DIR)
print("CLONING_DIR:", CLONING_DIR)
print("OUTPUT_BASE_DIR:", OUTPUT_BASE_DIR)
print("OUTPUTS_JSON_PATH:", OUTPUTS_JSON_PATH)
print("SPEAKER_CACHE_DIR:", SPEAKER_CACHE_DIR)

# Sanity checks
if not VIBEVOICE_DIR.exists():
    raise FileNotFoundError(f"VibeVoice repo not found at {VIBEVOICE_DIR}")

if not SPEAKERS_DIR.exists():
    raise FileNotFoundError(f"Speakers directory not found at {SPEAKERS_DIR}")

if not PROMPTS_DIR.exists():
    PROMPTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Created prompts directory at {PROMPTS_DIR}")

# Ensure cloning directory exists
CLONING_DIR.mkdir(parents=True, exist_ok=True)
print(f"Cloning directory ready at {CLONING_DIR}")

# Ensure outputs directory exists
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# Ensure speaker cache directory exists
SPEAKER_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Speaker cache directory ready at {SPEAKER_CACHE_DIR}")

# Editable install of the local VibeVoice repo from Drive (Colab only)
if IN_COLAB:
    # These are Colab magics and will only run in the notebook environment
    %cd $VIBEVOICE_DIR
    %pip install -e . --quiet
    %cd /content
else:
    print("Not running in Colab – assuming VibeVoice is already installed in this environment.")



Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/Libri_Vibevoice
VIBEVOICE_DIR: /content/drive/MyDrive/Libri_Vibevoice/VibeVoice
SPEAKERS_DIR: /content/drive/MyDrive/Libri_Vibevoice/speakers
PROMPTS_DIR: /content/drive/MyDrive/Libri_Vibevoice/prompts
CLONING_DIR: /content/drive/MyDrive/Libri_Vibevoice/cloning
OUTPUT_BASE_DIR: /content/drive/MyDrive/Libri_Vibevoice/outputs
OUTPUTS_JSON_PATH: /content/drive/MyDrive/Libri_Vibevoice/outputs/outputs.json
Cloning directory ready at /content/drive/MyDrive/Libri_Vibevoice/cloning
/content/drive/MyDrive/Libri_Vibevoice/VibeVoice
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
ERROR: Could not install packages due to an OSError: [Errno 107] Transport endpoint is not connected



ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-3603079909.py", line 58, in <cell line: 0>
    get_ipython().run_line_magic('cd', '/content')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most r

In [ ]:
# Cell 2 – Import VibeVoice and load the VibeVoice-1.5B model

import sys
import torch

# Ensure the local VibeVoice repo (VIBEVOICE_DIR) is on the Python path so imports work
# even when the working directory is /content in Colab.
sys.path.insert(0, str(VIBEVOICE_DIR))

from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

# Default model ID (change here if you want a different checkpoint)
MODEL_ID = "vibevoice/VibeVoice-1.5B"

# Simple device selection
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Using device: {DEVICE}")

# Load processor
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)

# Decide dtype and attention implementation
if DEVICE == "mps":
    load_dtype = torch.float32
    attn_impl_primary = "sdpa"
elif DEVICE == "cuda":
    load_dtype = torch.bfloat16
    attn_impl_primary = "flash_attention_2"
else:
    load_dtype = torch.float32
    attn_impl_primary = "sdpa"

print(f"torch_dtype: {load_dtype}, attn_implementation: {attn_impl_primary}")

# Load model with basic error handling around flash_attention_2
try:
    if DEVICE == "mps":
        model = VibeVoiceForConditionalGenerationInference.from_pretrained(
            MODEL_ID,
            torch_dtype=load_dtype,
            attn_implementation=attn_impl_primary,
            device_map=None,
        )
        model.to("mps")
    elif DEVICE == "cuda":
        model = VibeVoiceForConditionalGenerationInference.from_pretrained(
            MODEL_ID,
            torch_dtype=load_dtype,
            attn_implementation=attn_impl_primary,
            device_map="cuda",
        )
    else:
        model = VibeVoiceForConditionalGenerationInference.from_pretrained(
            MODEL_ID,
            torch_dtype=load_dtype,
            attn_implementation=attn_impl_primary,
            device_map="cpu",
        )
except Exception as e:
    if attn_impl_primary == "flash_attention_2":
        print("Error loading with flash_attention_2, falling back to sdpa:", e)
        model = VibeVoiceForConditionalGenerationInference.from_pretrained(
            MODEL_ID,
            torch_dtype=load_dtype,
            attn_implementation="sdpa",
            device_map=(DEVICE if DEVICE in ("cuda", "cpu") else None),
        )
        if DEVICE == "mps":
            model.to("mps")
    else:
        raise

model.eval()
model.set_ddpm_inference_steps(num_steps=10)
print("Model loaded and ready for inference.")



Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

Error loading with flash_attention_2, falling back to sdpa: FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package flash_attn seems to be not installed. Please refer to the documentation of https://huggingface.co/docs/transformers/perf_infer_gpu_one#flashattention-2 to install Flash Attention 2.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded and ready for inference.


In [ ]:
# Cell 3 – Scan LibriTTS-R-style speakers tree, pick the longest .wav per speaker,
# and copy them to the cloning directory for efficient reuse

import soundfile as sf
import shutil
from typing import Dict

from pathlib import Path


def _wav_duration_sec(path: Path) -> float:
    """Return duration in seconds for a .wav file."""
    audio, sr = sf.read(str(path))
    n_samples = audio.shape[0]
    return float(n_samples) / float(sr)


def load_speaker_reference_map_from_cloning(cloning_dir: Path) -> Dict[str, Path]:
    """Load speaker reference map directly from cloning directory.

    This is faster than scanning the speakers directory, but only works
    if files have already been copied to the cloning directory.

    Args:
        cloning_dir: Directory containing reference audio files named {speaker_id}.wav

    Returns:
        Dictionary mapping speaker_id to Path of reference file in cloning_dir.
    """
    mapping: Dict[str, Path] = {}

    if not cloning_dir.exists():
        return mapping

    for wav_file in sorted(cloning_dir.glob("*.wav")):
        speaker_id = wav_file.stem  # filename without extension
        if wav_file.is_file():
            mapping[speaker_id] = wav_file

    return mapping


def build_speaker_reference_map(speakers_root: Path, cloning_dir: Path, force_rescan: bool = False) -> Dict[str, Path]:
    """Build {speaker_id: longest_wav_path} for all speakers under speakers_root.

    Also copies the reference files to the cloning directory for efficient reuse.
    The returned mapping points to files in the cloning directory.

    If cloning directory already has files and force_rescan is False, will attempt
    to use existing files first, then scan only for missing speakers.

    Args:
        speakers_root: Root directory containing speaker subdirectories.
        cloning_dir: Directory where reference audio files will be copied.
        force_rescan: If True, always scan speakers_root even if cloning_dir has files.

    Returns:
        Dictionary mapping speaker_id to Path of reference file in cloning_dir.
    """
    mapping: Dict[str, Path] = {}
    summary_rows = []
    cloning_dir.mkdir(parents=True, exist_ok=True)

    # Try to load existing files from cloning directory first (faster)
    if not force_rescan:
        existing_mapping = load_speaker_reference_map_from_cloning(cloning_dir)
        if existing_mapping:
            print(f"Found {len(existing_mapping)} existing reference files in cloning directory.")
            # We'll still scan to check for new speakers or update existing ones
            # but we can use existing files as a starting point

    for speaker_dir in sorted(p for p in speakers_root.iterdir() if p.is_dir()):
        speaker_id = speaker_dir.name
        longest_path = None
        longest_dur = 0.0

        # Recursively walk all chapters and wav files
        for wav_path in speaker_dir.rglob("*.wav"):
            try:
                dur = _wav_duration_sec(wav_path)
            except Exception as e:  # Skip corrupted or unreadable files
                print(f"[WARN] Skipping {wav_path} due to error: {e}")
                continue

            if dur > longest_dur:
                longest_dur = dur
                longest_path = wav_path

        if longest_path is not None:
            # Copy the reference file to cloning directory
            cloning_file_path = cloning_dir / f"{speaker_id}.wav"

            # Only copy if the file doesn't exist or if the source is newer
            if not cloning_file_path.exists() or longest_path.stat().st_mtime > cloning_file_path.stat().st_mtime:
                shutil.copy2(longest_path, cloning_file_path)
                print(f"Copied reference audio for speaker {speaker_id} to {cloning_file_path}")
            else:
                print(f"Using existing reference audio for speaker {speaker_id} at {cloning_file_path}")

            mapping[speaker_id] = cloning_file_path
            summary_rows.append((speaker_id, longest_dur, longest_path, cloning_file_path))

    # Pretty-print a compact table
    summary_rows.sort(key=lambda r: r[0])
    print(f"\nFound {len(mapping)} speakers with reference audio.")
    print(f"{'Speaker':>8} | {'Seconds':>8} | {'Original Path':<50} | {'Cloning Path'}")
    print("-" * 120)
    for sid, dur, orig_path, clone_path in summary_rows:
        orig_str = str(orig_path)[:48] + ".." if len(str(orig_path)) > 50 else str(orig_path)
        print(f"{sid:>8} | {dur:8.2f} | {orig_str:<50} | {clone_path}")

    return mapping


# Build speaker reference map - will copy files to cloning directory
# Set force_rescan=True to always re-scan the speakers directory
FORCE_RESCAN = False  # Set to True if you want to re-scan and update all reference files

SPEAKER_REF_MAP = build_speaker_reference_map(SPEAKERS_DIR, CLONING_DIR, force_rescan=FORCE_RESCAN)

print("\nExample speaker IDs:", sorted(SPEAKER_REF_MAP.keys())[:10])
print(f"\nAll reference files are now in: {CLONING_DIR}")
print("Future runs will use these files directly from the cloning directory.")
print("Set FORCE_RESCAN=True in this cell to re-scan and update reference files if needed.")



Using existing reference audio for speaker 1447 at /content/drive/MyDrive/Libri_Vibevoice/cloning/1447.wav
Using existing reference audio for speaker 196 at /content/drive/MyDrive/Libri_Vibevoice/cloning/196.wav
Using existing reference audio for speaker 2196 at /content/drive/MyDrive/Libri_Vibevoice/cloning/2196.wav
Using existing reference audio for speaker 254 at /content/drive/MyDrive/Libri_Vibevoice/cloning/254.wav
Using existing reference audio for speaker 27 at /content/drive/MyDrive/Libri_Vibevoice/cloning/27.wav
Using existing reference audio for speaker 2989 at /content/drive/MyDrive/Libri_Vibevoice/cloning/2989.wav
Using existing reference audio for speaker 3436 at /content/drive/MyDrive/Libri_Vibevoice/cloning/3436.wav
Using existing reference audio for speaker 426 at /content/drive/MyDrive/Libri_Vibevoice/cloning/426.wav
Using existing reference audio for speaker 5808 at /content/drive/MyDrive/Libri_Vibevoice/cloning/5808.wav
Using existing reference audio for speaker 7794

In [ ]:
# Cell 4 – clone_speaker_text: generate speech conditioned on a speaker's reference wav
# Includes caching of processed speaker embeddings for efficient reuse

import time
import uuid
from IPython.display import Audio, display
import numpy as np
import pickle
import hashlib


def extract_text_from_prompt(prompt_data: list) -> str:
    """Extract and combine all user content from a prompt's question structure.

    Args:
        prompt_data: List of turns, where each turn is a list of messages with role/content.

    Returns:
        Combined text from all user messages.
    """
    text_parts = []
    for turn in prompt_data:
        for message in turn:
            if message.get("role") == "user":
                content = message.get("content", "")
                if content:
                    text_parts.append(content.strip())
    return " ".join(text_parts)


def get_speaker_cache_path(speaker_id: str, ref_path: Path) -> Path:
    """Get the cache file path for a speaker's processed embeddings.

    Args:
        speaker_id: Speaker ID string
        ref_path: Path to the reference audio file

    Returns:
        Path to the cache file
    """
    # Create a hash based on speaker_id and file modification time to detect changes
    file_mtime = ref_path.stat().st_mtime if ref_path.exists() else 0
    cache_key = f"{speaker_id}_{file_mtime}"
    cache_hash = hashlib.md5(cache_key.encode()).hexdigest()[:16]
    return SPEAKER_CACHE_DIR / f"{speaker_id}_{cache_hash}.pkl"


def load_cached_speaker_embeddings(speaker_id: str, ref_path: Path) -> dict | None:
    """Load cached speaker embeddings if they exist and are valid.

    Args:
        speaker_id: Speaker ID string
        ref_path: Path to the reference audio file

    Returns:
        Dictionary with cached embeddings, or None if not found/invalid
    """
    cache_path = get_speaker_cache_path(speaker_id, ref_path)

    if not cache_path.exists():
        return None

    try:
        with open(cache_path, 'rb') as f:
            cached_data = pickle.load(f)

        # Verify the cache is for the correct speaker and file
        if cached_data.get('speaker_id') == speaker_id and cached_data.get('ref_path') == str(ref_path):
            return cached_data
        else:
            # Cache is stale, remove it
            cache_path.unlink()
            return None
    except Exception as e:
        print(f"Warning: Error loading cache for speaker {speaker_id}: {e}")
        # Remove corrupted cache
        if cache_path.exists():
            cache_path.unlink()
        return None


def save_cached_speaker_embeddings(speaker_id: str, ref_path: Path, embeddings_data: dict):
    """Save processed speaker embeddings to cache.

    Args:
        speaker_id: Speaker ID string
        ref_path: Path to the reference audio file
        embeddings_data: Dictionary containing processed embeddings
    """
    cache_path = get_speaker_cache_path(speaker_id, ref_path)

    # Add metadata to the cache
    cache_data = {
        'speaker_id': speaker_id,
        'ref_path': str(ref_path),
        'embeddings': embeddings_data,
        'cached_at': time.time(),
    }

    try:
        with open(cache_path, 'wb') as f:
            pickle.dump(cache_data, f)
        print(f"Cached embeddings for speaker {speaker_id} at {cache_path}")
    except Exception as e:
        print(f"Warning: Error saving cache for speaker {speaker_id}: {e}")


def process_speaker_for_cloning(speaker_id: str, ref_path: Path) -> dict:
    """Process a speaker's reference audio to extract embeddings for cloning.
    Uses cache if available, otherwise processes and caches.

    Args:
        speaker_id: Speaker ID string
        ref_path: Path to the reference audio file

    Returns:
        Dictionary containing processed voice embeddings ready for use
    """
    # Check cache first
    cached = load_cached_speaker_embeddings(speaker_id, ref_path)
    if cached is not None:
        print(f"Using cached embeddings for speaker {speaker_id}")
        return cached['embeddings']

    # Process the speaker audio for the first time
    print(f"Processing speaker {speaker_id} for the first time (this may take a moment)...")

    # Use processor's internal method to extract voice embeddings
    # We need to call _create_voice_prompt directly to get just the voice parts
    voice_samples = [str(ref_path)]

    # Call the internal method to process voice samples
    voice_tokens, voice_speech_inputs, voice_speech_masks = processor._create_voice_prompt(voice_samples)

    # Extract and cache the voice-related embeddings
    # Convert numpy arrays to lists for pickle compatibility
    embeddings_data = {
        'voice_tokens': voice_tokens,
        'voice_speech_inputs': voice_speech_inputs,
        'voice_speech_masks': voice_speech_masks,
        'all_speakers': [0],  # Single speaker with ID 0
    }

    # Cache the embeddings
    save_cached_speaker_embeddings(speaker_id, ref_path, embeddings_data)

    return embeddings_data


def clone_speaker_text(
    speaker_id: str,
    text: str,
    prompt_id: str | None = None,
    batch_run_id: str | None = None,
    cfg_scale: float = 1.3,
    disable_prefill: bool = False,
    save_audio: bool = True,
):
    """Generate audio for plain text, conditioned on the chosen LibriTTS-R speaker.

    - speaker_id: e.g. "196" (must exist in SPEAKER_REF_MAP).
    - text: plain text; this function hides the "Speaker 0: ..." formatting.
    - prompt_id: ID of the prompt (used for filename).
    - batch_run_id: ID for the entire batch run (used for directory structure).
    - cfg_scale: CFG guidance scale.
    - disable_prefill: if True, turn off voice cloning (is_prefill=False).
    - save_audio: if False, skip saving audio file (for testing).

    Returns:
        dict with keys: 'output_path', 'duration', 'gen_time'
    """
    if not text or not text.strip():
        raise ValueError("Text is empty – please provide some content to synthesize.")

    if speaker_id not in SPEAKER_REF_MAP:
        available = ", ".join(sorted(SPEAKER_REF_MAP.keys()))
        raise KeyError(f"Unknown speaker_id '{speaker_id}'. Available: {available}")

    ref_path = SPEAKER_REF_MAP[speaker_id]

    # Model expects "Speaker X: ..." style internally; hide this from the user
    formatted_text = f"Speaker 0: {text.strip()}"

    # Check if we have cached embeddings for this speaker
    cached_data = load_cached_speaker_embeddings(speaker_id, ref_path)

    if cached_data is not None:
        # Use cached embeddings - manually reconstruct inputs to avoid reprocessing audio
        print(f"Using cached embeddings for speaker {speaker_id} (skipping audio processing)")
        cached_embeddings = cached_data['embeddings']

        # Parse the text to extract speaker and content
        # The formatted_text is "Speaker 0: {text}", so we parse it
        parsed_lines = processor._parse_script(formatted_text)
        all_speakers = list(set(speaker_id for speaker_id, _ in parsed_lines))

        # Create system prompt tokens
        system_tokens = processor.tokenizer.encode(processor.system_prompt)

        # Use cached voice tokens and embeddings
        voice_tokens = cached_embeddings['voice_tokens']
        voice_speech_inputs = cached_embeddings['voice_speech_inputs']
        voice_speech_masks = cached_embeddings['voice_speech_masks']

        # Build full token sequence (same as processor._process_single does)
        full_tokens = system_tokens + voice_tokens
        speech_input_mask = [False] * len(system_tokens) + voice_speech_masks

        # Add text input section
        text_input_tokens = processor.tokenizer.encode(' Text input:\n', add_special_tokens=False)
        full_tokens += text_input_tokens
        speech_input_mask += [False] * len(text_input_tokens)

        # Add parsed speaker lines
        for speaker_id_parsed, speaker_text in parsed_lines:
            speaker_text_tokens = processor.tokenizer.encode(f" Speaker {speaker_id_parsed}:{speaker_text}\n", add_special_tokens=False)
            full_tokens += speaker_text_tokens
            speech_input_mask += [False] * len(speaker_text_tokens)

        # Add speech output section
        speech_output_tokens = processor.tokenizer.encode(' Speech output:\n', add_special_tokens=False) + [processor.tokenizer.speech_start_id]
        full_tokens += speech_output_tokens
        speech_input_mask += [False] * len(speech_output_tokens)

        # Create encoding dict (similar to processor._process_single output)
        encoding = {
            "input_ids": full_tokens,
            "speech_inputs": voice_speech_inputs if voice_speech_inputs else None,
            "speech_input_mask": speech_input_mask,
            "parsed_script": parsed_lines,
            "all_speakers": all_speakers,
        }

        # Batch encode (handles padding and tensor conversion)
        inputs = processor._batch_encode(
            [encoding],
            padding=True,
            truncation=False,
            return_tensors="pt",
            return_attention_mask=True,
        )
    else:
        # Process speaker for the first time (will also cache)
        print(f"Processing speaker {speaker_id} for the first time (processing audio)...")
        cached_embeddings = process_speaker_for_cloning(speaker_id, ref_path)

        # Process the full input with voice samples (normal path)
        voice_samples = [[str(ref_path)]]
        inputs = processor(
            text=[formatted_text],
            voice_samples=voice_samples,
            padding=True,
            return_tensors="pt",
            return_attention_mask=True,
        )

    target_device = DEVICE if DEVICE in ("cuda", "mps") else "cpu"
    for k, v in inputs.items():
        if torch.is_tensor(v):
            inputs[k] = v.to(target_device)
        elif isinstance(v, list) and len(v) > 0 and torch.is_tensor(v[0]):
            # Handle lists of tensors (like voice_speech_inputs)
            inputs[k] = [t.to(target_device) if torch.is_tensor(t) else t for t in v]

    is_prefill = not disable_prefill


    print(
        f"Running generation for speaker {speaker_id} from {ref_path} "
        f"(is_prefill={is_prefill}, cfg_scale={cfg_scale})"
    )

    start_time = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=None,
        cfg_scale=cfg_scale,
        tokenizer=processor.tokenizer,
        generation_config={"do_sample": False},
        verbose=True,
        is_prefill=is_prefill,
    )
    gen_time = time.time() - start_time
    print(f"Generation time: {gen_time:.2f} seconds")

    speech = outputs.speech_outputs[0]
    if torch.is_tensor(speech):
        speech_np = speech.detach().cpu().float().numpy()
    else:
        speech_np = np.asarray(speech)

    sample_rate = 24000  # VibeVoice default
    duration = speech_np.shape[-1] / float(sample_rate)
    print(f"Generated audio duration: {duration:.2f} seconds")

    result = {
        'duration': duration,
        'gen_time': gen_time,
        'output_path': None
    }

    if save_audio:
        # Build output path: outputs/raw/{batch_run_id}/{speaker_id}/{prompt_id}.wav
        if batch_run_id:
            speaker_output_dir = OUTPUT_BASE_DIR / batch_run_id / speaker_id
        else:
            # Fallback if batch_run_id not provided
            speaker_output_dir = OUTPUT_BASE_DIR / speaker_id

        speaker_output_dir.mkdir(parents=True, exist_ok=True)

        # Filename based on prompt_id if provided, otherwise use unique_run_id
        filename = f"{prompt_id}.wav"

        out_path = speaker_output_dir / filename
        processor.save_audio(speech, output_path=str(out_path))
        result['output_path'] = str(out_path)
        print(f"Saved synthesized audio to: {out_path}")

    # Inline playback in Colab / notebooks (only for first few to avoid clutter)
    # try:
    #     display(Audio(speech_np, rate=sample_rate))
    # except Exception as e:  # Fallback if audio widget fails
    #     print(f"Could not display audio widget: {e}")

    return result



In [ ]:
# Cell 5 – Load JSON prompts and process all prompts with all speakers

import json
from typing import Dict, List, Any

def load_prompts_from_directory(prompts_dir: Path) -> List[Dict[str, Any]]:
    """Load all JSON prompt files from the prompts directory.

    Returns:
        List of prompt dictionaries, each with 'id' and 'question' keys.
    """
    prompts = []
    json_files = list(prompts_dir.glob("*.json"))

    if not json_files:
        print(f"Warning: No JSON files found in {prompts_dir}")
        return prompts

    for json_file in sorted(json_files):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                prompt_data = json.load(f)
                if 'id' in prompt_data and 'question' in prompt_data:
                    prompts.append(prompt_data)
                    print(f"Loaded prompt: {prompt_data['id']} from {json_file.name}")
                else:
                    print(f"Warning: {json_file.name} missing 'id' or 'question' keys, skipping")
        except json.JSONDecodeError as e:
            print(f"Error parsing {json_file.name}: {e}")
        except Exception as e:
            print(f"Error loading {json_file.name}: {e}")

    return prompts


def load_or_create_outputs_json(outputs_json_path: Path) -> Dict[str, Any]:
    """Load existing outputs.json or create a new structure.

    Returns:
        Dictionary with structure: {speaker_id: {prompt_id: {run_id: {...}}}}
    """
    if outputs_json_path.exists():
        try:
            with open(outputs_json_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"Error loading outputs.json: {e}, creating new file")

    return {}


def save_outputs_json(outputs_json_path: Path, outputs_data: Dict[str, Any]):
    """Save outputs data to JSON file."""
    outputs_json_path.parent.mkdir(parents=True, exist_ok=True)
    with open(outputs_json_path, 'w', encoding='utf-8') as f:
        json.dump(outputs_data, f, indent=2, ensure_ascii=False)
    print(f"Saved outputs to {outputs_json_path}")


def process_all_prompts_with_all_speakers(
    prompts: List[Dict[str, Any]],
    speakers: Dict[str, Path],
    cfg_scale: float = 1.3,
    disable_prefill: bool = False,
):
    """Process all prompts with all speakers and save results to outputs.json.

    Args:
        prompts: List of prompt dictionaries.
        speakers: Dictionary mapping speaker_id to reference audio path.
        cfg_scale: CFG guidance scale.
        disable_prefill: Whether to disable voice cloning.
    """
    import uuid

    # Generate a unique batch run ID for this entire processing session
    batch_run_id = str(uuid.uuid4())[:8]
    print(f"\n{'='*80}")
    print(f"Starting new batch run with ID: {batch_run_id}")
    print(f"Files will be saved to: {OUTPUT_BASE_DIR / batch_run_id}")
    print(f"{'='*80}")

    outputs_data = load_or_create_outputs_json(OUTPUTS_JSON_PATH)

    total_runs = len(prompts) * len(speakers)
    current_run = 0

    print(f"\nProcessing {len(prompts)} prompts with {len(speakers)} speakers")
    print(f"Total runs: {total_runs}\n")

    for prompt in prompts:
        prompt_id = prompt['id']
        question_data = prompt['question']

        # Extract text from prompt
        text = extract_text_from_prompt(question_data)
        print(f"\n{'='*80}")
        print(f"Processing prompt: {prompt_id}")
        print(f"Text: {text[:100]}..." if len(text) > 100 else f"Text: {text}")
        print(f"{'='*80}\n")

        # Initialize speaker entry if needed
        for speaker_id in sorted(speakers.keys()):
            current_run += 1
            print(f"[{current_run}/{total_runs}] Speaker: {speaker_id}, Prompt: {prompt_id}")

            # Generate audio
            result = clone_speaker_text(
                speaker_id=speaker_id,
                text=text,
                prompt_id=prompt_id,
                batch_run_id=batch_run_id,
                cfg_scale=cfg_scale,
                disable_prefill=disable_prefill,
                save_audio=True,
            )

            # Update outputs structure: {speaker_id: {prompt_id: {run_id: {...}}}}
            if speaker_id not in outputs_data:
                outputs_data[speaker_id] = {}

            if prompt_id not in outputs_data[speaker_id]:
                outputs_data[speaker_id][prompt_id] = {}

            # Store run information
            outputs_data[speaker_id][prompt_id] = {
                'audio_file': result['output_path'],
                'batch_run_id': batch_run_id,
                'duration': result['duration'],
                'generation_time': result['gen_time'],
                'cfg_scale': cfg_scale,
                'disable_prefill': disable_prefill,
                'prompt_text': text,
            }

            # Save after each run (in case of interruption)
            save_outputs_json(OUTPUTS_JSON_PATH, outputs_data)
            print(f"✓ Saved run for speaker {speaker_id}, prompt {prompt_id}\n")

    print(f"\n{'='*80}")
    print(f"Completed all {total_runs} runs!")
    print(f"Batch run ID: {batch_run_id}")
    print(f"Files saved to: {OUTPUT_BASE_DIR / batch_run_id}")
    print(f"Outputs metadata saved to: {OUTPUTS_JSON_PATH}")
    print(f"{'='*80}")


# Load prompts from directory
all_prompts = load_prompts_from_directory(PROMPTS_DIR)

if not all_prompts:
    print("\nNo prompts found. Please add JSON files to the prompts directory.")
    print("Example files have been created: multi_turn_base_19.json and simple_java_0.json")
else:
    # Configuration
    CFG_SCALE = 1.3
    DISABLE_PREFILL = False

    # Process all prompts with all speakers
    process_all_prompts_with_all_speakers(
        prompts=all_prompts,
        speakers=SPEAKER_REF_MAP,
        cfg_scale=CFG_SCALE,
        disable_prefill=DISABLE_PREFILL,
    )



Loaded prompt: multi_turn_base_19 from multi_turn_base_19.json
Loaded prompt: simple_java_0 from simple_java_0.json
Error parsing tmp.json: Extra data: line 2 column 1 (char 206)

Starting new batch run with ID: 25094444
Files will be saved to: /content/drive/MyDrive/Libri_Vibevoice/outputs/25094444

Processing 2 prompts with 10 speakers
Total runs: 20


Processing prompt: multi_turn_base_19
Text: Find every file with the name 'test_document.txt' nestled within the current directory. Transfer a d...

[1/20] Speaker: 1447, Prompt: multi_turn_base_19
Running generation for speaker 1447 from /content/drive/MyDrive/Libri_Vibevoice/cloning/1447.wav (is_prefill=True, cfg_scale=1.3)


Generating (active: 1/1):  31%|███       | 154/494 [00:34<01:14,  4.59it/s]

Samples [0] reached EOS token at step 156.


Generation time: 35.00 seconds
Generated audio duration: 20.53 seconds
Saved synthesized audio to: /content/drive/MyDrive/Libri_Vibevoice/outputs/25094444/1447/multi_turn_base_19.wav
Saved outputs to /content/drive/MyDrive/Libri_Vibevoice/outputs/outputs.json
✓ Saved run for speaker 1447, prompt multi_turn_base_19

[2/20] Speaker: 196, Prompt: multi_turn_base_19
Running generation for speaker 196 from /content/drive/MyDrive/Libri_Vibevoice/cloning/196.wav (is_prefill=True, cfg_scale=1.3)


Generating (active: 1/1):  30%|██▉       | 176/594 [00:39<01:31,  4.55it/s]

...

Samples [0] reached EOS token at step 99.


Generation time: 21.99 seconds
Generated audio duration: 12.93 seconds
Saved synthesized audio to: /content/drive/MyDrive/Libri_Vibevoice/outputs/25094444/5808/simple_java_0.wav
Saved outputs to /content/drive/MyDrive/Libri_Vibevoice/outputs/outputs.json
✓ Saved run for speaker 5808, prompt simple_java_0

[20/20] Speaker: 7794, Prompt: simple_java_0
Running generation for speaker 7794 from /content/drive/MyDrive/Libri_Vibevoice/cloning/7794.wav (is_prefill=True, cfg_scale=1.3)


Generating (active: 1/1):  22%|██▏       | 113/512 [00:25<01:28,  4.53it/s]

Samples [0] reached EOS token at step 115.


Generation time: 25.58 seconds
Generated audio duration: 15.07 seconds
Saved synthesized audio to: /content/drive/MyDrive/Libri_Vibevoice/outputs/25094444/7794/simple_java_0.wav
Saved outputs to /content/drive/MyDrive/Libri_Vibevoice/outputs/outputs.json
✓ Saved run for speaker 7794, prompt simple_java_0


Completed all 20 runs!
Batch run ID: 25094444
Files saved to: /content/drive/MyDrive/Libri_Vibevoice/outputs/25094444
Outputs metadata saved to: /content/drive/MyDrive/Libri_Vibevoice/outputs/outputs.json


In [ ]:
# Cell 6 – Helper: list all speakers and pre-process them for caching

from typing import Dict

def preprocess_all_speakers(speaker_map: Dict[str, Path], force_reprocess: bool = False):
    """Pre-process all speakers to cache their embeddings.

    This is useful to do once at the start, so all subsequent generations
    can use cached embeddings and skip audio processing.

    Args:
        speaker_map: Dictionary mapping speaker_id to reference audio path
        force_reprocess: If True, reprocess even if cache exists
    """
    print(f"\n{'='*80}")
    print(f"Pre-processing {len(speaker_map)} speakers for caching...")
    print(f"{'='*80}\n")

    processed = 0
    cached = 0

    for speaker_id in sorted(speaker_map.keys()):
        ref_path = speaker_map[speaker_id]

        if force_reprocess:
            # Force reprocessing by removing cache
            cache_path = get_speaker_cache_path(speaker_id, ref_path)
            if cache_path.exists():
                cache_path.unlink()

        # Check if already cached
        cached_data = load_cached_speaker_embeddings(speaker_id, ref_path)
        if cached_data is not None:
            print(f"✓ Speaker {speaker_id}: Already cached")
            cached += 1
        else:
            # Process and cache
            print(f"Processing speaker {speaker_id}...")
            try:
                process_speaker_for_cloning(speaker_id, ref_path)
                processed += 1
                print(f"✓ Speaker {speaker_id}: Processed and cached\n")
            except Exception as e:
                print(f"✗ Error processing speaker {speaker_id}: {e}\n")

    print(f"\n{'='*80}")
    print(f"Pre-processing complete!")
    print(f"  - Processed: {processed}")
    print(f"  - Already cached: {cached}")
    print(f"  - Total: {len(speaker_map)}")
    print(f"{'='*80}\n")


print("Listing all speakers and their reference wavs:\n")
for sid in sorted(SPEAKER_REF_MAP.keys()):
    print(f"Speaker {sid}: {SPEAKER_REF_MAP[sid]}")

print("\n" + "="*80)
print("To pre-process all speakers for faster subsequent runs, run:")
print("  preprocess_all_speakers(SPEAKER_REF_MAP)")
print("="*80)



Listing all speakers and their reference wavs:

Speaker 1447: /content/drive/MyDrive/Libri_Vibevoice/cloning/1447.wav
Speaker 196: /content/drive/MyDrive/Libri_Vibevoice/cloning/196.wav
Speaker 2196: /content/drive/MyDrive/Libri_Vibevoice/cloning/2196.wav
Speaker 254: /content/drive/MyDrive/Libri_Vibevoice/cloning/254.wav
Speaker 27: /content/drive/MyDrive/Libri_Vibevoice/cloning/27.wav
Speaker 2989: /content/drive/MyDrive/Libri_Vibevoice/cloning/2989.wav
Speaker 3436: /content/drive/MyDrive/Libri_Vibevoice/cloning/3436.wav
Speaker 426: /content/drive/MyDrive/Libri_Vibevoice/cloning/426.wav
Speaker 5808: /content/drive/MyDrive/Libri_Vibevoice/cloning/5808.wav
Speaker 7794: /content/drive/MyDrive/Libri_Vibevoice/cloning/7794.wav
